# DermNet EfficientNet-B3 Training for DermaSoul

This notebook trains an EfficientNet-B3 model on the DermNet dataset and saves a checkpoint in a format compatible with the app's disease model.

Important:
- The app expects 22 output classes.
- The class names must match the labels in `backend/app/ml/class_labels.py`.
- The saved checkpoint must be a PyTorch `state_dict()`.
- This notebook is for training only and does not change the project files.

## Goal
Reach around 85% validation accuracy on a correctly mapped DermNet validation set.

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision pillow scikit-learn matplotlib

import os
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b3
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1) Set dataset paths and class mapping

Update the `CLASS_MAP` so it matches the actual folder names in your DermNet dataset.

The app expects these labels exactly:

- Acne
- Actinic Keratosis
- Benign Tumors
- Bullous
- Candidiasis
- Drug Eruption
- Eczema
- Infestations/Bites
- Lichen
- Lupus
- Moles
- Psoriasis
- Rosacea
- Seborrheic Keratoses
- Skin Cancer
- Sun/Sunlight Damage
- Tinea
- Unknown/Normal
- Vascular Tumors
- Vasculitis
- Vitiligo
- Warts

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/shubhamgoel27/dermnet"
OUTPUT_DIR = "/kaggle/working"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "diagnosis_model_optimized.pth")

DATASET_CANDIDATES = []
for candidate in [DATASET_ROOT, os.path.join(DATASET_ROOT, "train"), os.path.join(DATASET_ROOT, "test")]:
    if os.path.exists(candidate):
        DATASET_CANDIDATES.append(candidate)

print("Dataset candidates:", DATASET_CANDIDATES)

PROJECT_LABELS = [
    "Acne",
    "Actinic Keratosis",
    "Benign Tumors",
    "Bullous",
    "Candidiasis",
    "Drug Eruption",
    "Eczema",
    "Infestations/Bites",
    "Lichen",
    "Lupus",
    "Moles",
    "Psoriasis",
    "Rosacea",
    "Seborrheic Keratoses",
    "Skin Cancer",
    "Sun/Sunlight Damage",
    "Tinea",
    "Unknown/Normal",
    "Vascular Tumors",
    "Vasculitis",
    "Vitiligo",
    "Warts",
]

LABEL_TO_INDEX = {label: idx for idx, label in enumerate(PROJECT_LABELS)}

CLASS_MAP = {
    "acne": "Acne",
    "actinic_keratosis": "Actinic Keratosis",
    "benign_tumors": "Benign Tumors",
    "bullous": "Bullous",
    "candidiasis": "Candidiasis",
    "drug_eruption": "Drug Eruption",
    "eczema": "Eczema",
    "infestations_bites": "Infestations/Bites",
    "lichen": "Lichen",
    "lupus": "Lupus",
    "moles": "Moles",
    "psoriasis": "Psoriasis",
    "rosacea": "Rosacea",
    "seborrheic_keratoses": "Seborrheic Keratoses",
    "skin_cancer": "Skin Cancer",
    "sun_damage": "Sun/Sunlight Damage",
    "tinea": "Tinea",
    "unknown_normal": "Unknown/Normal",
    "vascular_tumors": "Vascular Tumors",
    "vasculitis": "Vasculitis",
    "vitiligo": "Vitiligo",
    "warts": "Warts",
}

print(f"Dataset root: {DATASET_ROOT}")
print(f"Expected classes: {len(PROJECT_LABELS)}")

if os.path.exists(DATASET_ROOT):
    folders = sorted([
        d for d in os.listdir(DATASET_ROOT)
        if os.path.isdir(os.path.join(DATASET_ROOT, d))
    ])
    print("Actual dataset folders:")
    for d in folders:
        print(f" - {d}")
    print("Normalized folder keys:")
    for d in folders:
        print(f" - {d} -> {d.strip().lower().replace(' ', '_').replace('-', '_')}")

print("CLASS_MAP keys:")
print(CLASS_MAP.keys())


### Important: real DermNet folder names do not match the app labels

The dataset folders are grouped by broad DermNet categories such as:
- `Acne and Rosacea Photos`
- `Vascular Tumors`
- `Seborrheic Keratoses and other Benign Tumors`

Those are not the same as the project label names such as `Acne`, `Rosacea`, or `Benign Tumors`.

That is why the mapping must be done explicitly with `CLASS_MAP`, and why only one class may survive if the folder names are not normalized correctly.

In [ ]:
def normalize_key(name: str) -> str:
    name = name.strip().lower()
    name = name.replace(" ", "_")
    name = name.replace("-", "_")
    return name


def collect_samples(data_root: str):
    samples = []
    if not os.path.exists(data_root):
        raise FileNotFoundError(f"Dataset folder not found: {data_root}")

    search_roots = [data_root]
    train_dir = os.path.join(data_root, "train")
    test_dir = os.path.join(data_root, "test")
    if os.path.isdir(train_dir):
        search_roots = [train_dir]
    elif os.path.isdir(test_dir):
        search_roots = [test_dir]

    for search_root in search_roots:
        for class_dir in sorted(os.listdir(search_root)):
            full_dir = os.path.join(search_root, class_dir)
            if not os.path.isdir(full_dir):
                continue

            mapped = CLASS_MAP.get(normalize_key(class_dir))
            if mapped is None:
                continue

            for file_name in sorted(os.listdir(full_dir)):
                if file_name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
                    image_path = os.path.join(full_dir, file_name)
                    samples.append((image_path, mapped))

    if not samples:
        print("No valid samples found.")
        print("These are the actual top-level folders in your dataset:")
        for d in sorted(os.listdir(data_root)):
            full_dir = os.path.join(data_root, d)
            if os.path.isdir(full_dir):
                print(f" - {d} -> {normalize_key(d)}")
        raise ValueError(
            "No valid samples found. Update CLASS_MAP to match the real DermNet folder names shown above."
        )

    return samples

all_samples = collect_samples(DATASET_ROOT)
print(f"Total usable samples: {len(all_samples)}")
print(f"First sample: {all_samples[0]}")


## 2) Create train/validation split

We will split the dataset with a fixed random seed so results are reproducible.

Important diagnostic: if the dataset contains only one class after mapping, the validation accuracy will look artificially perfect because the model is only seeing one label.

In [ ]:
random.shuffle(all_samples)
val_count = max(1, int(len(all_samples) * 0.15))
train_samples = all_samples[val_count:]
val_samples = all_samples[:val_count]

print(f"Train samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")

from collections import Counter

val_counter = Counter(label for _, label in val_samples)
print("Validation class counts:")
print(val_counter)

dataset_counter = Counter()
for img_path, label_name in all_samples:
    dataset_counter[label_name] += 1

print("Dataset class counts:")
print(dataset_counter)

In [ ]:
class DermNetProjectDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label_name = self.samples[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label_index = LABEL_TO_INDEX[label_name]
        return image, label_index

# Same preprocessing as the app
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = DermNetProjectDataset(train_samples, transform=train_transform)
val_dataset = DermNetProjectDataset(val_samples, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train and validation loaders created.")

## 3) Build the model

This uses EfficientNet-B3 and keeps the output size at 22 to match the project.

In [ ]:
# Use a custom classifier head because the app expects exactly 22 disease classes.
# Loading pretrained ImageNet weights is fine only when the classifier stays at 1000 classes.
model = efficientnet_b3(weights=None, num_classes=len(PROJECT_LABELS))
model = model.to(DEVICE)
print(model)

In [ ]:
# Compute class weights to help with imbalance
from collections import Counter

counter = Counter(label for _, label in all_samples)
# Counter uses string labels, so convert to index first
index_counter = Counter(LABEL_TO_INDEX[label_name] for _, label_name in all_samples)

class_counts = [index_counter.get(i, 1) for i in range(len(PROJECT_LABELS))]
max_count = max(class_counts)
weights = torch.tensor([max_count / c if c > 0 else 1.0 for c in class_counts], dtype=torch.float32)
weights = weights.to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", weights)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## 4) Train the model

This is the core training loop.

In [ ]:
EPOCHS = 20
best_val_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_pred = []
    train_true = []

    for images, targets in train_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_pred.extend(logits.argmax(dim=1).cpu().tolist())
        train_true.extend(targets.cpu().tolist())

    train_loss /= len(train_dataset)
    train_acc = accuracy_score(train_true, train_pred)

    model.eval()
    val_pred = []
    val_true = []
    val_loss = 0.0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)

            val_pred.extend(logits.argmax(dim=1).cpu().tolist())
            val_true.extend(targets.cpu().tolist())

    val_loss /= len(val_dataset)
    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(val_true, val_pred, average="macro", zero_division=0)

    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Save the best checkpoint in a format compatible with the project
os.makedirs(OUTPUT_DIR, exist_ok=True)
if best_state is not None:
    torch.save(best_state, MODEL_SAVE_PATH)
    print(f"Saved best model to: {MODEL_SAVE_PATH}")
else:
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Saved final model to: {MODEL_SAVE_PATH}")

with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w") as f:
    json.dump({
        "labels": PROJECT_LABELS,
        "num_classes": len(PROJECT_LABELS),
        "best_val_accuracy": float(best_val_acc),
        "class_map": CLASS_MAP,
    }, f, indent=2)

print("Metadata saved.")

## 5) Evaluate final model

This prints final accuracy and per-class metrics.

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(targets.cpu().tolist())

final_acc = accuracy_score(all_targets, all_preds)
final_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
print(f"Final validation accuracy: {final_acc:.4f}")
print(f"Final macro F1: {final_f1:.4f}")

cm = confusion_matrix(all_targets, all_preds)
print("Confusion matrix shape:", cm.shape)

# Per-class recall
for idx, label in enumerate(PROJECT_LABELS):
    if (np.array(all_targets) == idx).sum() == 0:
        continue
    recall = ((np.array(all_targets) == idx) & (np.array(all_preds) == idx)).sum() / ((np.array(all_targets) == idx).sum())
    print(f"{label}: recall={recall:.4f}")

## 6) How to use this notebook

1. Open Kaggle.
2. Upload this notebook or paste its cells into a new Kaggle notebook.
3. Change `DATASET_ROOT` to the DermNet path.
4. Update `CLASS_MAP` to match the actual folder names in your dataset.
5. Run all cells from top to bottom.
6. Wait for training to finish.
7. The best checkpoint will be saved to `/kaggle/working/diagnosis_model_optimized.pth`.
8. Download the file and place it into your project model folder if you want to replace the current model.

### Important safety note
This notebook only trains and evaluates a model. It does not modify the project itself. You should only replace the app model after you confirm:
- class count is 22,
- label order matches the app,
- validation accuracy is acceptable,
- the checkpoint is saved as `state_dict()`.

## 7) Target results

A good target for your project is:
- 85%+ validation accuracy: good
- 90%+ validation accuracy: strong
- below 75%: likely not reliable enough yet

If the model underperforms, the usual causes are:
- wrong class mapping
- too many classes cut out or missing
- inconsistent preprocessing
- too little data for some classes
- no class weighting

This notebook is designed to help you reach the 85% goal while keeping compatibility with the app.